## 1. setup

- Imports numerical and data manipulation libraries, visualization packages, cross-validation handlers, standard performance metrics, and pre-processing pipeline modules.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print("Setup Complete")

## 2. Data Loading & Initial Missing Value Labeling

- Reads the house price dataset, drops records missing target values, transforms prices to log scale, and fills semantic missing values (None or 0) based on physical meaning.


In [ ]:
# dataset
X_full = pd.read_csv(
    "/kaggle/input/competitions/home-data-for-ml-course/train.csv", index_col="Id"
)
X_test_full = pd.read_csv(
    "/kaggle/input/competitions/home-data-for-ml-course/test.csv", index_col="Id"
)

# Delete rows with missing target values to ensure baseline quality
X_full.dropna(axis=0, subset=["SalePrice"], inplace=True)

# Apply log1p transformation to the target column to normalize the target distribution
y_log = np.log1p(X_full.SalePrice)
X_full.drop(["SalePrice"], axis=1, inplace=True)

# Fill categorical missing values representing the total absence of a physical facility
none_cols = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
]
for col in none_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].fillna("None")
        X_test_full[col] = X_test_full[col].fillna("None")

# Fill continuous missing values representing a missing component with a zero value
zero_cols = [
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
]
for col in zero_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].fillna(0)
        X_test_full[col] = X_test_full[col].fillna(0)

# Check matrix dimensions
print(f"Train shape: {X_full.shape}")
print(f"Test shape: {X_test_full.shape}")

""" >>>
Train shape: (1460, 79)
Test shape: (1459, 79)
"""

## 3. Feature Engineering with Interaction Terms

- Generates standard structural engineered terms (Total SF, Age) and adds powerful interaction terms (OverallQual \* GrLivArea, etc.) to improve predictions.


In [ ]:
def add_custom_features(df):
    df_out = df.copy()

    # Recast MSSubClass feature type to object for categorical processing
    if "MSSubClass" in df_out.columns:
        df_out["MSSubClass"] = df_out["MSSubClass"].astype(str)

    # Calculate total living space area
    df_out["TotalSF"] = df_out["TotalBsmtSF"] + df_out["1stFlrSF"] + df_out["2ndFlrSF"]

    # Compute Total Bathroom Counts (combining full and half bath)
    df_out["TotalBaths"] = (
        df_out["FullBath"]
        + (0.5 * df_out["HalfBath"])
        + df_out["BsmtFullBath"]
        + (0.5 * df_out["BsmtHalfBath"])
    )

    # Calculate Outdoor Total Deck and Porch Area
    df_out["TotalOutsideSF"] = (
        df_out["WoodDeckSF"]
        + df_out["OpenPorchSF"]
        + df_out["EnclosedPorch"]
        + df_out["3SsnPorch"]
        + df_out["ScreenPorch"]
    )

    # Calculate structural age and remodeling elapsed years
    df_out["HouseAge"] = df_out["YrSold"] - df_out["YearBuilt"]
    df_out["RemodAge"] = df_out["YrSold"] - df_out["YearRemodAdd"]

    # Engineer powerful interaction features to boost regression accuracy
    df_out["OverallQual_GrLivArea"] = df_out["OverallQual"] * df_out["GrLivArea"]
    df_out["OverallQual_TotalSF"] = df_out["OverallQual"] * df_out["TotalSF"]
    df_out["GarageScore"] = df_out["GarageCars"] * df_out["GarageArea"]

    return df_out


# Run custom feature generation workflows
X_full_fe = add_custom_features(X_full)
X_test_full_fe = add_custom_features(X_test_full)

## 4. Feature Selection & Grouping

- Categorizes features into ordinal strings, nominal category columns, and mathematical numerical values to fit specialized transformers.


In [ ]:
# Select qualitative features that follow an inherent ranking
ordinal_cols = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
]

# Map specific rating progression (from Po=Poor to Ex=Excellent)
qual_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ordinal_categories = [qual_order for _ in ordinal_cols]

# Select nominal categorical columns that do not possess strict rankings
categorical_cols = [
    cname
    for cname in X_full_fe.columns
    if X_full_fe[cname].dtype == "object" and cname not in ordinal_cols
]

# Isolate numerical continuous and discrete feature columns
numerical_cols = [
    cname
    for cname in X_full_fe.columns
    if X_full_fe[cname].dtype in ["int64", "float64"]
]

# Consolidate target feature subset columns and extract matrices
my_cols = numerical_cols + ordinal_cols + categorical_cols
X = X_full_fe[my_cols].copy()
X_test = X_test_full_fe[my_cols].copy()

print(f"Selected numerical features: {len(numerical_cols)}")
print(f"Selected categorical features: {len(categorical_cols)}")

""" >>>
Selected numerical features: 43
Selected categorical features: 35
"""

## 5. Preprocessing Pipelines (Tree vs. Linear Models)

- Shares categorical transformations, but splits numerical branches into a basic imputation tree-branch and an un-biased scaled linear-branch.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Shared Block: Categorical ordinal pipeline handling sequential ranks
ord_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# Shared Block: Categorical nominal pipeline executing One-Hot Encoding
cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Distinct Branch: Specialized numerical transformation optimized for Tree-based models (no scaling)
num_transformer_tree = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", num_transformer_tree, numerical_cols),
        ("ord", ord_transformer, ordinal_cols),
        ("cat", cat_transformer, categorical_cols),
    ]
)

# Distinct Branch: Specialized numerical transformation optimized for Linear models (requires scaling)
num_transformer_linear = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)
preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", num_transformer_linear, numerical_cols),
        ("ord", ord_transformer, ordinal_cols),
        ("cat", cat_transformer, categorical_cols),
    ]
)

## 6. Global Configuration & RMSLE Evaluator

- Defines a centralized configuration class for cross-validation splits and XGBoost hyperparameters, and implements an RMSLE evaluation function.


In [ ]:
from sklearn.metrics import root_mean_squared_log_error


# Centralized training configuration parameters
class Config:
    N_SPLITS = 10
    RANDOM_STATE = 42
    SHUFFLE = True
    XGB_PARAMS = {
        "n_estimators": 1500,
        "learning_rate": 0.05,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "n_jobs": -1,
    }


# Evaluates the Root Mean Squared Log Error between targets and predictions
def evaluate_rmsle(y_true_log, y_pred_log):
    mse = mean_squared_error(y_true_log, y_pred_log)
    return np.sqrt(mse)

## 7. Linear Model Group (Lasso / ElasticNet / Ridge) OOF Function

- Implements a 10-fold Out-Of-Fold (OOF) cross-validation framework to train, predict, and evaluate standard regularized linear regression models.


In [ ]:
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.model_selection import KFold


# Trains linear models and generates Out-of-Fold predictions and test predictions
def run_linear_oof_training(X, y_log, X_test, preprocessor):
    models = {
        "Lasso": Lasso(alpha=0.0005, max_iter=10000, random_state=42),
        "ElasticNet": ElasticNet(
            alpha=0.0005, l1_ratio=0.5, max_iter=10000, random_state=42
        ),
        "Ridge": Ridge(alpha=12),
    }

    # Initialize zero arrays for tracking training predictions and test metrics
    oof_preds = {name: np.zeros(len(X)) for name in models}
    test_preds = {name: np.zeros(len(X_test)) for name in models}
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )

    print(f"Starting Linear Model Group {Config.N_SPLITS}-Fold OOF Cross Validation")

    # Execute the cross-validation loop across partitions
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        # Fit and transform the preprocessing pipeline on current data
        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        # Train each model configuration and aggregate fold outputs
        for name, model in models.items():
            model.fit(X_train_trans, y_train)
            oof_preds[name][val_idx] = model.predict(X_val_trans)
            test_preds[name] += model.predict(X_test_trans) / Config.N_SPLITS

    # Calculate overall validation scores for each linear model
    for name in models:
        score = evaluate_rmsle(y_log, oof_preds[name])
        print(f"{name} Overall OOF RMSLE: {score:.5f}")

    return oof_preds, test_preds

## 8. XGBoost Model OOF Function

- Executes a parallel 10-fold OOF cross-validation loop tailored for the tree-based XGBoost model, tracking validation error logs per fold.


In [ ]:
from sklearn.model_selection import KFold
import numpy as np
from xgboost import XGBRegressor


# Trains the XGBoost model using an Out-of-Fold cross-validation strategy
def run_xgb_oof_training(X, y_log, X_test, preprocessor):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )

    print(f"Starting XGBoost Model {Config.N_SPLITS}-Fold OOF Cross Validation")

    # Iterate through each defined cross-validation partition
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        # Transform structural variables utilizing tree-specific configurations
        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        # Initialize XGBRegressor using config hyperparameters
        model = XGBRegressor(**Config.XGB_PARAMS)
        model.fit(
            X_train_trans, y_train, eval_set=[(X_val_trans, y_val)], verbose=False
        )

        # Log out-of-fold validation estimations
        oof_preds[val_idx] = model.predict(X_val_trans)
        test_preds += model.predict(X_test_trans) / Config.N_SPLITS

        # Calculate metric errors for the active running fold
        fold_score = evaluate_rmsle(y_val, oof_preds[val_idx])
        print(f"-> Fold {fold + 1} RMSLE: {fold_score:.5f}")

    # Evaluate aggregate system error benchmarks across the full array
    overall_score = evaluate_rmsle(y_log, oof_preds)
    print(f"XGBoost Overall OOF RMSLE: {overall_score:.5f}\n")
    
    return oof_preds, test_preds

## 9. Execution & Weighted Model Ensemble

- Runs both tree and linear model functions, then performs a weighted blend ensemble (70% XGBoost, 15% Lasso, 15% Ridge) to minimize overall error.


In [ ]:
# Execute training for tree-based architectures
oof_xgb, test_preds_xgb = run_xgb_oof_training(X, y_log, X_test, preprocessor_tree)

# Execute training for regularized linear pipelines
oof_linears, test_preds_linears = run_linear_oof_training(
    X, y_log, X_test, preprocessor_linear
)

# Combine models into a weighted blend ensemble
final_oof_blend = (
    (oof_xgb * 0.70) + (oof_linears["Lasso"] * 0.15) + (oof_linears["Ridge"] * 0.15)
)
blend_cv_score = evaluate_rmsle(y_log, final_oof_blend)
print(f"Ensemble Blended Final OOF RMSLE: {blend_cv_score:.5f}")

""" >>>
開始 XGBoost 模型 10-Fold OOF 交叉驗證
-> Fold 1 RMSLE: 0.10823
-> Fold 2 RMSLE: 0.14419
-> Fold 3 RMSLE: 0.10205
-> Fold 4 RMSLE: 0.13562
-> Fold 5 RMSLE: 0.15259
-> Fold 6 RMSLE: 0.11807
-> Fold 7 RMSLE: 0.13381
-> Fold 8 RMSLE: 0.10652
-> Fold 9 RMSLE: 0.12298
-> Fold 10 RMSLE: 0.08253
XGBoost Overall OOF RMSLE: 0.12236

開始線性模型群 10-Fold OOF 交叉驗證
Lasso Overall OOF RMSLE: 0.13836
ElasticNet Overall OOF RMSLE: 0.13754
Ridge Overall OOF RMSLE: 0.13836
Ensemble Blended Final OOF RMSLE: 0.12009
"""

## 10. Test Inference & CSV Submission Export

- Computes the final ensemble predictions on the test set, reverses the log scale back to actual dollar figures, and saves the formatted submission file.


In [ ]:
# Blend individual test predictions using corresponding model weights
final_test_preds_log = (
    (test_preds_xgb * 0.70)
    + (test_preds_linears["Lasso"] * 0.15)
    + (test_preds_linears["Ridge"] * 0.15)
)

# Convert log scale values back to original price metrics
final_preds_dollar = np.expm1(final_test_preds_log)

print("Are there any nulls in test predictions?: ", np.isnan(final_preds_dollar).any())
print(
    "Descriptive statistics for predicted prices:\\n",
    pd.Series(final_preds_dollar).describe(),
)

""" >>>
Are there any nulls in test predictions?: False
Descriptive statistics for predicted prices:
 count      1459.000000
mean     177436.145275
std       74944.688911
min       46079.553175
25%      127157.730807
50%      156253.672658
75%      209793.341282
max      536456.281016
dtype: float64
"""

In [ ]:
# Construct the output file structure for final submission
output = pd.DataFrame({"Id": X_test.index, "SalePrice": final_preds_dollar})

output.to_csv("submission.csv", index=False)
print("submission.csv successfully exported! Format as follows:")
print(output.head())

""" >>>
submission.csv successfully exported! Format as follows:
     Id      SalePrice
0  1461  119718.201737
1  1462  161059.028979
2  1463  180148.245048
3  1464  194390.327671
4  1465  186170.532748
"""